# 01 - ImageNet-100 Data Pipeline

This notebook sets up the data pipeline for ImageNet-100:
- Download ImageNet-100 dataset (~14GB, 130k images, 100 classes)
- Organize into train/val folders
- Define RGB noise functions for self-healing
- Create DataLoaders optimized for RTX 3050

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
import zipfile
import shutil
from pathlib import Path

# Set base directory
BASE_DIR = r"C:\Users\Rushikesh\OneDrive\CODES\SelfHealingNN"
os.chdir(BASE_DIR)

# Device setup
torch.set_num_threads(8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Configuration

In [ ]:
# Dataset configuration
IMAGE_SIZE = 224
BATCH_SIZE = 32  # Optimized for RTX 3050 (4GB VRAM) with 224x224 RGB
NUM_WORKERS = 4
NUM_CLASSES = 100

# Paths
DATA_DIR = os.path.join(BASE_DIR, "imagenet100_data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data directory: {DATA_DIR}")

## 2. Download ImageNet-100

**Option 1: Kaggle (Recommended)**
```bash
pip install kaggle
kaggle datasets download -d ambityga/imagenet100
```

**Option 2: Manual Download**
1. Go to: https://www.kaggle.com/datasets/ambityga/imagenet100
2. Download and extract to `imagenet100_data/`

**Option 3: HuggingFace**
```python
from datasets import load_dataset
dataset = load_dataset("imagenet-100")
```

In [ ]:
# Check if data already exists
def check_dataset():
    if os.path.exists(TRAIN_DIR) and os.path.exists(VAL_DIR):
        train_classes = len(os.listdir(TRAIN_DIR))
        val_classes = len(os.listdir(VAL_DIR))
        if train_classes >= 100 and val_classes >= 100:
            print(f"Dataset already exists!")
            print(f"Train classes: {train_classes}")
            print(f"Val classes: {val_classes}")
            return True
    return False

if not check_dataset():
    print("Dataset not found. Please download using one of these methods:")
    print("")
    print("Method 1 - Kaggle CLI:")
    print("  pip install kaggle")
    print("  kaggle datasets download -d ambityga/imagenet100")
    print("  # Extract to imagenet100_data/")
    print("")
    print("Method 2 - Manual:")
    print("  1. Visit: https://www.kaggle.com/datasets/ambityga/imagenet100")
    print("  2. Download and extract to:", DATA_DIR)

In [ ]:
# Helper function to extract Kaggle download
def extract_kaggle_download(zip_path, extract_to):
    """Extract Kaggle downloaded zip file."""
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraction complete!")
    
# Uncomment and run if you downloaded via Kaggle:
# extract_kaggle_download("imagenet100.zip", DATA_DIR)

## 3. Data Transforms

**Important:** No normalization for VAE (keep [0,1] range for sigmoid output)

In [ ]:
# Training transforms with augmentation
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),  # Converts to [0, 1] range
    # NO Normalize - VAE needs [0, 1] for sigmoid output
])

# Validation/Test transforms (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

print("Transforms defined (no normalization for VAE compatibility)")

## 4. Create DataLoaders

In [ ]:
# Create datasets
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_transforms)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True  # Consistent batch sizes
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Train: {len(train_dataset):,} images ({len(train_loader)} batches)")
print(f"Val: {len(val_dataset):,} images ({len(val_loader)} batches)")
print(f"Classes: {len(train_dataset.classes)}")

In [ ]:
# Show class distribution
class_names = train_dataset.classes
print(f"\nFirst 20 classes:")
for i, name in enumerate(class_names[:20]):
    print(f"  {i:3d}: {name}")
print(f"  ... and {len(class_names) - 20} more")

## 5. RGB Noise Functions

These functions corrupt images to test self-healing capability.

In [ ]:
def add_gaussian_noise(image_tensor, noise_factor=0.3):
    """
    Add Gaussian noise to RGB images.
    
    Args:
        image_tensor: (B, 3, H, W) in [0, 1] range
        noise_factor: Standard deviation of noise (0.0 to 1.0)
    
    Returns:
        Noisy tensor clamped to [0, 1]
    """
    noise = torch.randn_like(image_tensor) * noise_factor
    return torch.clamp(image_tensor + noise, 0., 1.)


def add_salt_pepper_noise(image_tensor, amount=0.05):
    """
    Add salt and pepper noise.
    
    Args:
        image_tensor: (B, 3, H, W) in [0, 1] range
        amount: Proportion of pixels to corrupt (0.0 to 1.0)
    """
    noisy = image_tensor.clone()
    
    # Salt (white pixels)
    salt_mask = torch.rand_like(image_tensor) < (amount / 2)
    noisy[salt_mask] = 1.0
    
    # Pepper (black pixels)
    pepper_mask = torch.rand_like(image_tensor) < (amount / 2)
    noisy[pepper_mask] = 0.0
    
    return noisy


def add_speckle_noise(image_tensor, noise_factor=0.3):
    """
    Add multiplicative speckle noise.
    Common in radar/ultrasound images.
    """
    noise = torch.randn_like(image_tensor) * noise_factor
    return torch.clamp(image_tensor + image_tensor * noise, 0., 1.)


def add_mixed_noise(image_tensor, gaussian_factor=0.2, sp_amount=0.02):
    """
    Add combination of Gaussian and salt-pepper noise.
    More realistic corruption.
    """
    noisy = add_gaussian_noise(image_tensor, gaussian_factor)
    noisy = add_salt_pepper_noise(noisy, sp_amount)
    return noisy


# Default noise function for training
def add_noise(image_tensor, noise_factor=0.3):
    """Default noise function (Gaussian)."""
    return add_gaussian_noise(image_tensor, noise_factor)


print("Noise functions defined:")
print("  - add_gaussian_noise(tensor, noise_factor=0.3)")
print("  - add_salt_pepper_noise(tensor, amount=0.05)")
print("  - add_speckle_noise(tensor, noise_factor=0.3)")
print("  - add_mixed_noise(tensor, gaussian_factor=0.2, sp_amount=0.02)")
print("  - add_noise(tensor, noise_factor=0.3)  [default]")

## 6. Visualize Sample Images

In [ ]:
def show_images(images, titles=None, nrow=4, figsize=(16, 4)):
    """Display a grid of images."""
    n = min(len(images), nrow)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    
    for i, (img, ax) in enumerate(zip(images[:n], axes)):
        # Convert from (C, H, W) to (H, W, C)
        if isinstance(img, torch.Tensor):
            img = img.cpu().numpy()
        if img.shape[0] == 3:  # CHW -> HWC
            img = np.transpose(img, (1, 2, 0))
        img = np.clip(img, 0, 1)
        
        ax.imshow(img)
        ax.axis('off')
        if titles:
            ax.set_title(titles[i], fontsize=10)
    
    plt.tight_layout()
    plt.show()


# Get a batch of images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print(f"Value range: [{images.min():.2f}, {images.max():.2f}]")

# Show clean images
titles = [class_names[l] for l in labels[:4]]
show_images(images[:4], titles=titles)
plt.suptitle("Clean Images", y=1.02)

In [ ]:
# Visualize noise effects
sample_img = images[0:1]  # Single image, keep batch dim

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Row 1: Different noise types
noise_types = [
    ("Clean", sample_img),
    ("Gaussian (0.3)", add_gaussian_noise(sample_img, 0.3)),
    ("Salt-Pepper (0.1)", add_salt_pepper_noise(sample_img, 0.1)),
    ("Mixed", add_mixed_noise(sample_img, 0.2, 0.03)),
]

for i, (title, img) in enumerate(noise_types):
    axes[0, i].imshow(np.transpose(img[0].numpy(), (1, 2, 0)))
    axes[0, i].set_title(title)
    axes[0, i].axis('off')

# Row 2: Different Gaussian noise levels
noise_levels = [0.1, 0.2, 0.3, 0.4]
for i, level in enumerate(noise_levels):
    noisy = add_gaussian_noise(sample_img, level)
    axes[1, i].imshow(np.transpose(noisy[0].numpy(), (1, 2, 0)))
    axes[1, i].set_title(f"Gaussian ({level})")
    axes[1, i].axis('off')

plt.suptitle("Noise Types and Levels", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Dataset Statistics

In [ ]:
# Compute mean and std of dataset (useful for reference)
def compute_mean_std(loader, num_batches=100):
    """Compute mean and std of dataset."""
    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_images = 0
    
    for i, (images, _) in enumerate(loader):
        if i >= num_batches:
            break
        batch_size = images.size(0)
        images = images.view(batch_size, 3, -1)  # (B, 3, H*W)
        mean += images.mean(dim=[0, 2]) * batch_size
        std += images.std(dim=[0, 2]) * batch_size
        total_images += batch_size
    
    mean /= total_images
    std /= total_images
    return mean, std

print("Computing dataset statistics (first 100 batches)...")
mean, std = compute_mean_std(train_loader)
print(f"Mean: [{mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f}]")
print(f"Std:  [{std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f}]")
print("\nNote: We don't normalize for VAE training (keep [0,1] range)")

## 8. Save Configuration

In [ ]:
# Configuration summary
config = {
    'dataset': 'ImageNet-100',
    'num_classes': NUM_CLASSES,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'train_samples': len(train_dataset),
    'val_samples': len(val_dataset),
    'channels': 3,
    'data_dir': DATA_DIR,
}

print("="*50)
print("ImageNet-100 Data Pipeline Ready!")
print("="*50)
for key, value in config.items():
    print(f"{key:15s}: {value}")
print("="*50)
print("\nNext: Run 02_ImageNet100_Healer_VAE.ipynb to train the healer!")